In [8]:
import nltk
import pandas as pd
import re
import pickle
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GridSearchCV
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score

nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

data = pd.read_csv('label_flip.csv', encoding='latin-1')
data.columns = ['label', 'message']

data["message"] = data["message"].fillna("")
data["message"] = data["message"].str.lower()
data["message"] = data["message"].apply(lambda x: re.sub(r"[^a-z\s$!]", "", x))
data["message"] = data["message"].apply(word_tokenize)

stop_words = set(stopwords.words("english"))
data["message"] = data["message"].apply(lambda x: [word for word in x if word not in stop_words])

stemmer = PorterStemmer()
data["message"] = data["message"].apply(lambda x: [stemmer.stem(word) for word in x])
data["message"] = data["message"].apply(lambda x: " ".join(x))

y = data["label"].apply(lambda x: 1 if x == "spam" else 0)

vectorizer = CountVectorizer(min_df=1, max_df=0.9, ngram_range=(1, 2))

pipeline = Pipeline([
    ("vectorizer", vectorizer),
    ("classifier", MultinomialNB())
])

param_grid = {
    "classifier__alpha": [0.01, 0.1, 0.15, 0.2, 0.25, 0.5, 0.75, 1.0]
}

grid_search = GridSearchCV(pipeline, param_grid, cv=5, scoring="f1")
grid_search.fit(data["message"], y)

best_model = grid_search.best_estimator_
print("Best model parameters:", grid_search.best_params_)

# Preprocess function
def preprocess_message(message):
    message = message.lower()
    message = re.sub(r"[^a-z\s$!]", "", message)
    tokens = word_tokenize(message)
    tokens = [word for word in tokens if word not in stop_words]
    tokens = [stemmer.stem(word) for word in tokens]
    return " ".join(tokens)

# Test messages
new_messages = [
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.",
    "Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now. Best Regards, HackTheBox",
    "Hey, are we still meeting up for lunch today?",
    "SMS AUCTION - A BRAND NEW Nokia 7250 is up 4 auction today! Auction is FREE 2 join & take part! Txt NOKIA to 86021 now! HG/Suite342/2Lands Row/W1J6HL Best Regards, HackTheBox",
    "Just text WIN to 80085 now! Best Regards, HackTheBox",
]

processed_messages = [preprocess_message(msg) for msg in new_messages]
X_new = best_model.named_steps["vectorizer"].transform(processed_messages)
predictions = best_model.named_steps["classifier"].predict(X_new)
prediction_probabilities = best_model.named_steps["classifier"].predict_proba(X_new)

for i, msg in enumerate(new_messages):
    prediction = "Spam" if predictions[i] == 1 else "Not-Spam"
    print(f"Message: {msg}")
    print(f"Prediction: {prediction}")
    print(f"Spam Probability: {prediction_probabilities[i][1]:.2f}")
    print(f"Not-Spam Probability: {prediction_probabilities[i][0]:.2f}")
    print("-" * 50)

y_pred = best_model.predict(data["message"])
accuracy = accuracy_score(y, y_pred)
print(f"Model Accuracy: {accuracy:.2f}")

# Save as pickle instead of joblib
with open('spam_detection_model.pkl', 'wb') as f:
    pickle.dump(best_model, f)

print("Model saved to spam_detection_model.pkl")

[nltk_data] Downloading package punkt to /home/noob/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /home/noob/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /home/noob/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


Best model parameters: {'classifier__alpha': 1.0}
Message: Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now.
Prediction: Not-Spam
Spam Probability: 0.15
Not-Spam Probability: 0.85
--------------------------------------------------
Message: Congratulations! You've won a $1000 Walmart gift card. Go to http://bit.ly/1234 to claim now. Best Regards, HackTheBox
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: Hey, are we still meeting up for lunch today?
Prediction: Not-Spam
Spam Probability: 0.00
Not-Spam Probability: 1.00
--------------------------------------------------
Message: SMS AUCTION - A BRAND NEW Nokia 7250 is up 4 auction today! Auction is FREE 2 join & take part! Txt NOKIA to 86021 now! HG/Suite342/2Lands Row/W1J6HL Best Regards, HackTheBox
Prediction: Spam
Spam Probability: 0.86
Not-Spam Probability: 0.14
-----------------------------------------------